# FAITH-Detect — Full experimental grid (Colab T4)

Runs the same `run_full_experiment` code as the local smoke run, scaled up: 5 seeds,
`roberta-base`, full training data, **RAID `reviews`** for cross-domain + cross-generator
evaluation, and **saved checkpoints** per variant.

Steps: GPU check -> deps -> upload code + MAiDE-up CSV -> download RAID `reviews` -> run ->
download results/figures -> (optional) download trained models.

> Runtime -> Change runtime type -> **T4 GPU** before running. If any cell OOMs, **Runtime ->
> Restart session** before retrying (a crashed run keeps its GPU memory until restart).

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 1) Dependencies.
!pip -q install captum statsmodels umap-learn shap lime datasets spacy huggingface_hub >/dev/null
!python -m spacy download en_core_web_sm -q
import nltk
for p in ['stopwords','punkt','punkt_tab','wordnet','omw-1.4','averaged_perceptron_tagger_eng']:
    nltk.download(p, quiet=True)
print('deps ready')

In [ ]:
# 2) Get the FAITH-Detect code (upload FAITH-Detect.zip). To force a fresh copy after an edit,
#    first run:  %cd /content  and  !rm -rf FAITH-Detect
import os, zipfile
from google.colab import files
if not os.path.exists('FAITH-Detect'):
    print('Upload FAITH-Detect.zip:')
    up = files.upload(); name = next(iter(up))
    with zipfile.ZipFile(name) as z: z.extractall('.')
%cd FAITH-Detect
import sys; sys.path.insert(0, 'src')

In [ ]:
# 3) MAiDE-up CSV -> ./data/all_data.csv
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/all_data.csv'):
    from google.colab import files; import shutil
    print('Upload all_data.csv:'); up = files.upload()
    shutil.move(next(iter(up)), 'data/all_data.csv')
print('CSV ready:', os.path.exists('data/all_data.csv'))

In [ ]:
# 4) RAID 'reviews' OOD (cross-domain same-task + cross-generator). The CSV is domain-sorted, so
#    we download it once and chunk-read it (early-stops after the reviews domain). The CSV is
#    large (tens of GB); download + scan take a while. Fast alternative: skip this cell and set
#    ood_domains=['abstracts'], ood_csv_path=None in cell 5 (streamed).
from huggingface_hub import hf_hub_download
from faithdetect.data import load_raid_from_csv
RAID_CSV = hf_hub_download('liamdugan/raid', 'train.csv', repo_type='dataset')
print('RAID CSV at', RAID_CSV)
ood = load_raid_from_csv(RAID_CSV, domains=('reviews',), cap_per_group=300,
                         cache_path='results/cache/raid_reviews.parquet')
print('reviews OOD:', len(ood), 'rows; generators:', sorted(ood['model'].unique()))

In [ ]:
# 5) Configure and run (5 seeds, roberta-base, reviews OOD + cross-generator, SAVE checkpoints).
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from faithdetect.experiment import ExperimentConfig, run_full_experiment
from faithdetect.utils.logging import save_json
from faithdetect.viz import make_all_figures

cfg = ExperimentConfig(
    name='full', data_csv='data/all_data.csv', encoder_name='roberta-base',
    variants=('baseline','hardmask','softreg'), seeds=(0,1,2,3,4),
    epochs=4, batch_size=16, max_length=256, train_subsample=None,
    softreg_lambda=0.5, xai_method='ig', ig_steps=50,
    faithfulness_n_texts=100, n_example_explanations=8,
    ood_csv_path=RAID_CSV, ood_domains=('reviews',), ood_cap_per_group=300,
    ood_cache='results/cache/raid_reviews.parquet', ood_per_generator=True,
    save_models=True, models_dir='models',   # persist the ref-seed checkpoint per variant
    device='cuda',
)
results = run_full_experiment(cfg)
save_json('results/full_results.json', results)
figs = make_all_figures(results, 'figures')
print('done:', len(figs), 'figures; saved models:', results.get('model_paths'))

In [ ]:
# 6) Results table + download (small: JSON + figures).
!python scripts/summarize_results.py --results results/full_results.json
!zip -qr faithdetect_outputs.zip results/full_results.json figures
from google.colab import files; files.download('faithdetect_outputs.zip')

In [ ]:
# 7) (Optional) Save the trained checkpoints. Each is ~500 MB, so 3 variants ~= 1.5 GB.
#    Option A: copy to Google Drive (recommended for large files).
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/FAITH-Detect-models && cp -v models/*.pt /content/drive/MyDrive/FAITH-Detect-models/
#    Option B: zip + browser download (slower for 1.5 GB):
# !zip -qr faithdetect_models.zip models
# from google.colab import files; files.download('faithdetect_models.zip')
#    Option C: push to the HuggingFace Hub (set a token first):
# from huggingface_hub import HfApi; api = HfApi()
# api.create_repo('scar09-22/faith-detect', repo_type='model', exist_ok=True)
# api.upload_folder(folder_path='models', repo_id='scar09-22/faith-detect', repo_type='model')

### Variations
- **Harder cross-domain shift:** `ood_domains=('reviews','news')`.
- **roberta-large ablation:** `encoder_name='roberta-large'` (keep `batch_size<=8`).
- **Save all seeds:** the run saves the reference seed per variant; to keep every seed, raise
  `save_models` handling in `experiment.py` (or run `scripts/run_grid.py`, which checkpoints cells).
- **Adversarial robustness:** RAID ships 11 attacks; load with `attacks=(...)` in
  `load_raid_from_csv` and evaluate as an extra OOD frame.